In [ ]:
import pandas as pd
import numpy as np

# 1. Load Data
raw_df = pd.read_csv('Atlantic_United_States.csv')

# 2. Data Validation & Formatting
raw_df['date'] = pd.to_datetime(raw_df['date'], format='%d-%m-%Y')
raw_df['duration_min'] = raw_df['duration_ms'] / 60000
raw_df['artist'] = raw_df['artist'].astype(str).str.strip()
raw_df['song'] = raw_df['song'].astype(str).str.strip()
raw_df['album_type'] = raw_df['album_type'].astype(str).str.title()
raw_df['is_explicit'] = raw_df['is_explicit'].astype(str).str.upper()
raw_df['rank_bucket'] = pd.cut(raw_df['position'], bins=[0, 10, 20, 50], labels=['Top 10', 'Top 20', 'Top 50'])

# 2a. Validation summary
validation_summary = {
    'missing_values': int(raw_df.isna().sum().sum()),
    'invalid_rank_rows': int(raw_df[~raw_df['position'].between(1, 50)].shape[0]),
    'duplicate_song_date_entries': int(raw_df.duplicated(subset=['song', 'artist', 'date']).sum())
}

# 3. Feature Engineering: Song-Level Metrics
raw_df = raw_df.sort_values(by=['song', 'date']).reset_index(drop=True)
raw_df['pop_trend_7d'] = raw_df.groupby('song')['popularity'].transform(lambda x: x.rolling(window=7, min_periods=1).mean())
raw_df['rank_change'] = raw_df.groupby('song')['position'].diff().fillna(0)

song_metrics = raw_df.groupby(['song', 'artist']).agg(
    days_on_chart=('date', 'count'),
    first_date=('date', 'min'),
    last_date=('date', 'max'),
    avg_rank=('position', 'mean'),
    best_rank=('position', 'min'),
    rank_volatility=('position', 'std'),
    avg_popularity=('popularity', 'mean'),
    pop_volatility=('popularity', 'std')
).reset_index()

song_metrics['rank_volatility'] = song_metrics['rank_volatility'].fillna(0)
song_metrics['pop_volatility'] = song_metrics['pop_volatility'].fillna(0)
song_metrics['chart_span_days'] = (song_metrics['last_date'] - song_metrics['first_date']).dt.days + 1

artist_metrics = raw_df.groupby('artist').agg(
    unique_songs=('song', 'nunique'),
    total_chart_days=('date', 'count'),
    avg_rank=('position', 'mean'),
    best_rank=('position', 'min'),
    avg_popularity=('popularity', 'mean')
).reset_index().sort_values(by='total_chart_days', ascending=False)

# 4. Leaderboards and trend overviews
longest_running = song_metrics.sort_values('days_on_chart', ascending=False).head(10)
top_popularity = song_metrics.sort_values('avg_popularity', ascending=False).head(10)
artist_dominance = artist_metrics.head(10)

print('Validation summary:')
print(validation_summary)
print('\nTop 10 longest charting songs:')
display(longest_running)
print('\nTop 10 songs by average popularity:')
display(top_popularity)
print('\nTop artists by total chart days:')
display(artist_dominance)


Song Metrics Snapshot:


,song,artist,days_on_chart,avg_rank,best_rank,rank_volatility
0,"""Slut!"" (Taylor's Version) (From The Vault)",Taylor Swift,26,15.500000,3,12.130128
1,(Don't Fear) The Reaper,Blue Öyster Cult,1,50.000000,50,0.000000
2,(There's No Place Like) Home for the Holidays ...,Perry Como,2,45.000000,45,0.000000
3,16 CARRIAGES,Beyoncé,10,26.000000,16,10.635371
4,2 hands,Tate McRae,7,33.571429,8,12.286268
